# Season Leaders audit

Season Leaders audit -- run after every parser run, from the app's repo root
(expects ./data/*.csv). Exits non-zero if any check fails.

Checks:
  1. Every played game in uww_schedule reconciles to uww_pbp_box_score
     (catches game_date_for() collapsing rematches onto the first meeting).
  2. uww_pbp_box_score "games" per player == real schedule meetings
     (catches nunique(opponent) being used as a game count).
  3. Team minutes per game ~= 200 (5 x 40) on both sides
     (catches the stint-clock inflation).
  4. No unclassified-PBP junk strings leaking in as player names.
  5. Opponent AST/STL/BLK denominators use the player's own games played.

In [9]:
import re, sys, os
import pandas as pd

# --- locate the parser's OUTPUT_DIR ---------------------------------------------------------------
# Set DATA_DIR explicitly if your layout differs, e.g.:
#   DATA_DIR = r"C:\Users\you\uww-app\data"
DATA_DIR = os.environ.get("DATA_DIR")

if DATA_DIR is None:
    for candidate in ["data", "../data", "../../data", ".",
                      os.path.join("..", "app", "data"), os.path.join("..", "streamlit", "data")]:
        if os.path.exists(os.path.join(candidate, "uww_schedule.csv")):
            DATA_DIR = candidate
            break

if DATA_DIR is None or not os.path.exists(os.path.join(DATA_DIR, "uww_schedule.csv")):
    raise FileNotFoundError(
        "Could not find uww_schedule.csv.\n"
        f"  Notebook is running in: {os.getcwd()}\n"
        f"  Looked in: data, ../data, ../../data, ., ../app/data, ../streamlit/data\n"
        "  Set DATA_DIR at the top of this cell to the folder holding the parser's CSV exports."
    )

print(f"Reading from {os.path.abspath(DATA_DIR)}\n")

fails = []
def check(ok, msg):
    print(("  PASS  " if ok else "  FAIL  ") + msg)
    if not ok:
        fails.append(msg)

def load(n):
    p = os.path.join(DATA_DIR, f"{n}.csv")
    if not os.path.exists(p):
        print(f"  (missing) {n}.csv")
        return pd.DataFrame()
    df = pd.read_csv(p)
    print(f"  {n}.csv  {len(df):>6} rows")
    return df

MONTHS = {"Jan":1,"Feb":2,"Mar":3,"Apr":4,"Aug":8,"Sep":9,"Oct":10,"Nov":11,"Dec":12}
def sched_date(d, start_year=2025):
    m = re.match(r"^\w{3},\s+(\w{3})\s+(\d+)$", str(d).strip())
    if not m:
        return None
    mo, dy = MONTHS[m.group(1)], int(m.group(2))
    return f"{start_year if mo >= 8 else start_year + 1}-{mo:02d}-{dy:02d}"

sched  = load("uww_schedule")
box    = load("uww_pbp_box_score")
prof   = load("uww_player_profiles")
prior  = load("uww_opponent_prior_games_pbp")

# Fail here, with a readable message, rather than as a bare KeyError three lines down.
for name, df, needed in [("uww_schedule", sched, "team"), ("uww_pbp_box_score", box, "team")]:
    if df.empty:
        raise ValueError(f"{name}.csv is empty or was not loaded -- check DATA_DIR above.")
    if needed not in df.columns:
        raise ValueError(f"{name}.csv has no {needed!r} column; found: {list(df.columns)}")

uww = sched[sched["team"].astype(str).str.contains("Whitewater", case=False, na=False)].copy()
played = uww[uww["outcome"].notna()].copy()
played["gd"] = played["date"].apply(sched_date)

print(f"\n{len(uww)} UWW games on the schedule, {len(played)} played")

Reading from C:\Users\frits\OneDrive\Documents\GitHub\uwwmensbball\data

  uww_schedule.csv     541 rows
  uww_pbp_box_score.csv     490 rows
  uww_player_profiles.csv     235 rows
  uww_opponent_prior_games_pbp.csv   14677 rows

28 UWW games on the schedule, 27 played


In [10]:
print("\n[1] box score reconciles to schedule finals")
side = box.groupby(["game_date", "team"])["PTS"].sum().reset_index()
mine = side[side["team"] == "UW-Whitewater"].set_index("game_date")["PTS"]
theirs = side[side["team"] != "UW-Whitewater"].groupby("game_date")["PTS"].sum()
missing = []
for _, r in played.iterrows():
    if r["gd"] not in mine.index:
        missing.append(f'{r["gd"]} {r["opponent"]}')
        continue
    got = (int(mine[r["gd"]]), int(theirs.get(r["gd"], 0)))
    want = (int(r["team_score"]), int(r["opponent_score"]))
    check(got == want, f'{r["gd"]} {r["opponent"][:26]:26} schedule {want[0]}-{want[1]} vs box {got[0]}-{got[1]}')
check(not missing, f"every played game present in box score (missing: {missing or 'none'})")


[1] box score reconciles to schedule finals
  PASS  2025-11-07 Ripon Red Hawks            schedule 76-58 vs box 76-58
  PASS  2025-11-14 St. Thomas (TX) Celts      schedule 73-81 vs box 73-81
  PASS  2025-11-15 Eureka Red Devils          schedule 116-73 vs box 116-73
  PASS  2025-11-19 Aurora Spartans            schedule 82-73 vs box 82-73
  PASS  2025-11-25 Simpson Storm              schedule 100-78 vs box 100-78
  PASS  2025-12-02 Elmhurst Bluejays          schedule 88-72 vs box 88-72
  PASS  2025-12-10 Lawrence Vikings           schedule 88-58 vs box 88-58
  PASS  2025-12-13 Carroll (WI) Pioneers      schedule 64-60 vs box 64-60
  PASS  2025-12-19 Hope Flying Dutchmen       schedule 95-92 vs box 95-92
  PASS  2025-12-20 Alma Scots                 schedule 90-61 vs box 90-61
  PASS  2025-12-30 Coe Kohawks                schedule 73-86 vs box 73-86
  FAIL  2026-01-03 UW-Oshkosh Titans          schedule 78-90 vs box 179-181
  FAIL  2026-01-07 UW-Stevens Point Pointers  schedule 71-79 

In [11]:
print("\n[2] per-player game counts match real schedule meetings")
meet = played["opponent"].value_counts().to_dict()
w = box[box["team"] == "UW-Whitewater"].copy()
w["real"] = w["opponent"].map(meet).fillna(1)
per = w.groupby("player").agg(app=("opponent", "nunique"), real=("real", "sum"))
bad = per[per["app"] != per["real"]]
check(bad.empty, f"nunique(opponent) == real games for all players ({len(bad)} mismatched)")
if not bad.empty:
    print(bad.head(5).to_string())


[2] per-player game counts match real schedule meetings
  FAIL  nunique(opponent) == real games for all players (20 mismatched)
                 app  real
player                    
Austin Ambrose     8    13
Brock Marino      17    26
Collin Madson     18    27
Darius Chestnut   16    21
Isaac Verges      17    26


In [12]:
print("\n[3] team minutes ~200 per game")
for label, df, tcol in [("UWW box", box, "team")]:
    if "MIN" in df.columns:
        g = df.groupby(["game_date", tcol])["MIN"].sum()
        off = g[(g < 150) | (g > 260)]
        check(off.empty, f"{label}: all team-games within 150-260 total min ({len(off)} outliers)")
        if not off.empty:
            print(off.head(8).to_string())


[3] team minutes ~200 per game
  FAIL  UWW box: all team-games within 150-260 total min (14 outliers)
game_date   team                     
2026-01-03  UW-Oshkosh Titans            4428.3
            UW-Whitewater                4535.4
2026-01-07  UW-Stevens Point Pointers    4902.1
            UW-Whitewater                4942.4
2026-01-10  UW-River Falls Falcons       3962.2
            UW-Whitewater                3940.7
2026-01-14  UW-La Crosse Eagles          9680.5
            UW-Whitewater                9528.8


In [13]:
print("\n[4] no unclassified PBP text leaking in as players")
PAT = re.compile(r"Commits|Turnover|Jump Ball|Subs In|Subs Out|Timeout|Rebound$", re.I)
for name, df, col in [("box", box, "player"), ("profiles", prof, "name"), ("prior_pbp", prior, "player")]:
    if df.empty or col not in df.columns:
        continue
    junk = sorted({p for p in df[col].dropna().unique() if PAT.search(str(p))})
    check(not junk, f"{name}: no junk player names ({junk[:4] or 'none'})")


[4] no unclassified PBP text leaking in as players
  FAIL  box: no junk player names (['Commits Foul', 'Jump Ball (Block Tie Up)'])
  FAIL  profiles: no junk player names (['Commits Foul', 'Damyen Jackson Commits Foul'])
  FAIL  prior_pbp: no junk player names (['Austin Bienemann Commits Foul', 'Commits Foul', 'Damyen Jackson Commits Foul'])


In [14]:
print("\n[5] opponent rate stats use each player's own games played")
if not prior.empty:
    upc = uww[uww.get("Upcoming", "No").astype(str).str.strip().str.lower() == "yes"]
    if not upc.empty:
        full = str(upc.iloc[0]["opponent"])
        short = next((t for t in prior["team"].dropna().unique() if str(t) in full or full.startswith(str(t))), None)
        if short:
            own = prior[prior["team"] == short]
            team_g = own["game_date"].nunique()
            pg = own.groupby("player")["game_date"].nunique()
            partial = pg[pg < team_g]
            check(partial.empty,
                  f"{short}: all players appeared in all {team_g} games "
                  f"({len(partial)} played fewer -> their AST/STL/BLK per-game are understated)")
            if not partial.empty:
                print(partial.sort_values().head(6).to_string())

print(f"\n{'ALL CHECKS PASSED' if not fails else str(len(fails)) + ' CHECK(S) FAILED'}")
sys.exit(1 if fails else 0)


[5] opponent rate stats use each player's own games played
  FAIL  Loras Duhawks: all players appeared in all 28 games (20 played fewer -> their AST/STL/BLK per-game are understated)
player
Damyen Jackson Commits Foul    1
Official TV                    3
Commits Foul                   3
Cooper Grams                   7
Juvon Crawford                 7
Ethan Meyer                    7

14 CHECK(S) FAILED


SystemExit: 1

C:\Users\frits\anaconda3\Lib\site-packages\IPython\core\interactiveshell.py:3513: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)
